# 01 — Data Acquisition & Coverage Analysis

**Constituency:** ชัยภูมิ เขต 2 · 341 polling stations · 4 อำเภอ (จัตุรัส, บ้านเขว้า, เนินสง่า, ซับใหญ่)

## What this notebook does

1. **Pull reference data** from P\'PanJ\'s curated GitHub (station list, candidates, parties, ground-truth totals)
2. **Verify** raw PDFs in `data/raw/pdfs_gdrive/`
3. **Dedupe** — Drive folder has every file copied twice; remove duplicates
4. **Coverage analysis** — match available PDFs against expected stations, identify gaps
5. **Build manifest** — single source of truth for what notebook 02 will OCR

## Coverage gap is a data-prep insight 📊

The Drive folder published by อ.จารย์ doesn\'t cover all 341 stations. We\'ll OCR what\'s available and document the gap explicitly — this becomes a finding for Component 4 (insights) and demonstrates real-world data quality challenges.

## Outputs

| File | Description |
|---|---|
| `data/external/stations.csv` | All 341 stations from ECT (reference) |
| `data/external/candidates.csv` | 7 candidates for ชัยภูมิ เขต 2 |
| `data/external/parties.csv` | 57 national parties |
| `data/external/reference_constituency.json` | Ground-truth totals (สส.6/1) |
| `data/external/coverage_report.csv` | What the Drive covers vs what\'s expected |
| `data/external/source_manifest.csv` | Source PDFs to feed into nb02 |


## 1 · Setup

In [1]:
# !pip install pandas requests pypdf --quiet

In [2]:
import os, json, hashlib, logging
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd
import requests
from pypdf import PdfReader

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s | %(message)s")
log = logging.getLogger("nb01")

# Paths
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

GDRIVE_DIR     = PROJECT_ROOT / "data" / "raw" / "pdfs_gdrive"
EXTERNAL_DIR   = PROJECT_ROOT / "data" / "external"
EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)

# Constituency metadata
PROVINCE_NAME      = "ชัยภูมิ"
PROVINCE_CODE      = "36"
CONSTITUENCY_NO    = 2
EXPECTED_STATIONS  = 341

print(f"Project root: {PROJECT_ROOT}")
print(f"PDF source:   {GDRIVE_DIR}")
print(f"External:     {EXTERNAL_DIR}")


Project root: d:\DSDE\ProjectDSDE_election2026
PDF source:   d:\DSDE\ProjectDSDE_election2026\data\raw\pdfs_gdrive
External:     d:\DSDE\ProjectDSDE_election2026\data\external


## 2 · Reference data from P\'PanJ

P\'PanJ (Pana Jirabunditsakul) curated election data on GitHub:
- `killernay/election-station-69` — every polling station in Thailand (94k+ stations)
- `killernay/election-69-OCR-result` — OCR\'d สส.6/1 (constituency-level summary) for every เขต

The constituency-level OCR gives us **canonical candidate/party rosters** plus **ground-truth totals** to validate our station-level OCR against.


In [3]:
def fetch_json(url, cache, force=False):
    """Fetch JSON with on-disk cache."""
    if cache.exists() and not force:
        log.info(f"Cached: {cache.name}")
        return json.loads(cache.read_text(encoding="utf-8"))
    log.info(f"Downloading: {url}")
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    cache.write_text(r.text, encoding="utf-8")
    return r.json()

# === 2.1 Stations ===
STATIONS_URL = "https://raw.githubusercontent.com/killernay/election-station-69/main/election-stations-2569.json"
all_stations = fetch_json(STATIONS_URL, EXTERNAL_DIR / "_raw_stations.json")

target_stations = []
for prov in all_stations["provinces"]:
    if prov["name"] != PROVINCE_NAME:
        continue
    for area in prov["areas"]:
        if area["area"] != CONSTITUENCY_NO:
            continue
        for s in area["stations"]:
            target_stations.append({
                "station_code":  s["code"],
                "station_no":    s["code"].rsplit("-", 1)[-1],
                "station_name":  s["name"],
                "district":      s["district"],
                "subdistrict":   s["subdistrict"],
            })

assert len(target_stations) == EXPECTED_STATIONS, \
    f"Expected {EXPECTED_STATIONS}, got {len(target_stations)}"

stations_df = pd.DataFrame(target_stations).sort_values("station_code").reset_index(drop=True)
stations_df.to_csv(EXTERNAL_DIR / "stations.csv", index=False, encoding="utf-8-sig")
print(f"\n✓ Saved {len(stations_df)} stations -> stations.csv")
stations_df.groupby("district")["subdistrict"].nunique().to_frame("n_tambols").assign(
    n_stations=stations_df.groupby("district").size()
)


2026-05-08 10:41:41,033 INFO | Cached: _raw_stations.json



✓ Saved 341 stations -> stations.csv


,n_tambols,n_stations
district,,
จัตุรัส,9,145
ซับใหญ่,3,37
บ้านเขว้า,6,112
เนินสง่า,4,47


In [4]:
# === 2.2 Candidates + parties + ground truth ===
CONSTIT_URL   = f"https://raw.githubusercontent.com/killernay/election-69-OCR-result/main/data/matched/constituency/{PROVINCE_CODE}_{CONSTITUENCY_NO}.json"
PARTYLIST_URL = f"https://raw.githubusercontent.com/killernay/election-69-OCR-result/main/data/matched/party_list/{PROVINCE_CODE}_{CONSTITUENCY_NO}.json"

constit = fetch_json(CONSTIT_URL,   EXTERNAL_DIR / f"_raw_constit_{PROVINCE_CODE}_{CONSTITUENCY_NO}.json")
plist   = fetch_json(PARTYLIST_URL, EXTERNAL_DIR / f"_raw_plist_{PROVINCE_CODE}_{CONSTITUENCY_NO}.json")

# Candidates
candidates_df = pd.DataFrame([{
    "candidate_no":    r["number"],
    "candidate_name":  r["name"],
    "party_name":      r["party"],
    "votes_reference": r["votes"],
} for r in constit["results"]]).sort_values("candidate_no").reset_index(drop=True)
candidates_df.to_csv(EXTERNAL_DIR / "candidates.csv", index=False, encoding="utf-8-sig")

# Parties
parties_df = pd.DataFrame([{
    "party_no":   r["number"],
    "party_name": r["party"],
    "votes_reference_constituency": r["votes"],
} for r in plist["results"]]).sort_values("party_no").reset_index(drop=True)
parties_df.to_csv(EXTERNAL_DIR / "parties.csv", index=False, encoding="utf-8-sig")

# Ground truth bundle
ground_truth = {
    "constituency": {
        "summary_by_section":    constit["summary_by_section"],
        "candidates":            constit["results"],
        "total_candidate_votes": sum(r["votes"] for r in constit["results"]),
    },
    "party_list": {
        "summary_by_section": plist["summary_by_section"],
        "parties":            plist["results"],
        "total_party_votes":  sum(r["votes"] for r in plist["results"]),
    },
    "source": "killernay/election-69-OCR-result",
    "constituency_id": f"{PROVINCE_CODE}_{CONSTITUENCY_NO}",
}
(EXTERNAL_DIR / "reference_constituency.json").write_text(
    json.dumps(ground_truth, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"✓ {len(candidates_df)} candidates, {len(parties_df)} parties")
print(f"✓ Ground truth saved")
print()
print(f"Reference totals for ชัยภูมิ เขต 2:")
print(f"  Eligible voters:           {constit['summary_by_section'].get('1.1', 0):>10,}")
print(f"  Voters who showed up:      {constit['summary_by_section'].get('1.2', 0):>10,}")
print(f"  Total constituency votes:  {ground_truth['constituency']['total_candidate_votes']:>10,}")
print(f"  Total party-list votes:    {ground_truth['party_list']['total_party_votes']:>10,}")


2026-05-08 10:41:41,348 INFO | Cached: _raw_constit_36_2.json
2026-05-08 10:41:41,351 INFO | Cached: _raw_plist_36_2.json


✓ 7 candidates, 57 parties
✓ Ground truth saved

Reference totals for ชัยภูมิ เขต 2:
  Eligible voters:              133,561
  Voters who showed up:          87,136
  Total constituency votes:      79,326
  Total party-list votes:        79,374


## 3 · Dedupe Drive PDFs

The downloaded Drive folder has each file copied **twice** with identical names in the same parent folder. Different mechanism on Windows — possibly OneDrive sync or zip extraction quirk. We dedupe by keeping the first occurrence per `(parent_folder, filename_lowercase)` pair.

> **Verification step**: before deleting, we hash each file pair to confirm they\'re byte-identical. If they differ, we keep both and flag for review.


In [5]:
def dedupe_drive_folder(folder: Path, *, dry_run: bool = True):
    """
    Group files by (parent, filename.lower()).
    For each group with >1 file, hash the contents — if identical, delete extras.
    Returns DataFrame describing actions.
    """
    if not folder.exists():
        raise FileNotFoundError(f"{folder} does not exist")

    # Use case-insensitive glob (.pdf and .PDF both)
    all_pdfs = list(folder.rglob("*.pdf")) + list(folder.rglob("*.PDF"))
    # Dedupe paths themselves (rglob may return same path twice on case-insensitive FS)
    seen_paths = {}
    for p in all_pdfs:
        seen_paths[str(p).lower()] = p
    pdfs = list(seen_paths.values())

    # Group by (parent, lowercase name)
    groups = defaultdict(list)
    for p in pdfs:
        groups[(str(p.parent), p.name.lower())].append(p)

    actions = []
    for (parent, name_lower), paths in groups.items():
        if len(paths) == 1:
            actions.append({"action": "KEEP", "path": str(paths[0].relative_to(folder)),
                            "size_kb": paths[0].stat().st_size // 1024, "note": ""})
            continue

        # Hash each to verify identity
        hashes = {}
        for p in paths:
            h = hashlib.md5()
            with open(p, "rb") as f:
                while chunk := f.read(65536):
                    h.update(chunk)
            hashes[str(p)] = h.hexdigest()

        unique_hashes = set(hashes.values())
        if len(unique_hashes) == 1:
            # All identical — keep first, delete rest
            keeper = paths[0]
            actions.append({"action": "KEEP", "path": str(keeper.relative_to(folder)),
                            "size_kb": keeper.stat().st_size // 1024,
                            "note": f"original of {len(paths)} duplicates"})
            for extra in paths[1:]:
                if not dry_run:
                    extra.unlink()
                actions.append({"action": "DELETE" if not dry_run else "WOULD_DELETE",
                                "path": str(extra.relative_to(folder)),
                                "size_kb": extra.stat().st_size // 1024,
                                "note": "byte-identical duplicate"})
        else:
            # Different content — keep all and flag
            for p in paths:
                actions.append({"action": "REVIEW", "path": str(p.relative_to(folder)),
                                "size_kb": p.stat().st_size // 1024,
                                "note": f"differs from siblings, hash={hashes[str(p)][:8]}"})

    return pd.DataFrame(actions)


# Step 1: dry run — see what would happen
print("=" * 60)
print("DRY RUN (no files deleted)")
print("=" * 60)
preview = dedupe_drive_folder(GDRIVE_DIR, dry_run=True)
print(preview["action"].value_counts().to_string())
print()

needs_review = preview[preview["action"] == "REVIEW"]
if len(needs_review):
    print("⚠️  Files needing manual review (different content, same name):")
    print(needs_review.to_string(index=False))
else:
    print("✓ No suspicious duplicates — all duplicates are byte-identical, safe to delete")


DRY RUN (no files deleted)
action
KEEP    35

✓ No suspicious duplicates — all duplicates are byte-identical, safe to delete


In [6]:
# Step 2: actually dedupe (uncomment when ready)
# results = dedupe_drive_folder(GDRIVE_DIR, dry_run=False)
# results.to_csv(EXTERNAL_DIR / "dedupe_log.csv", index=False, encoding="utf-8-sig")
# print(f"Done — {(results['action'] == 'DELETE').sum()} files removed")


## 4 · Classify available files

Walk `pdfs_gdrive/` (after dedupe) and figure out **what each PDF is**:

- **Constituency-level files** at root: 5/16, 5/17 (advance voting forms)
- **Tambol-level files**: contains all stations of one ตำบล stacked together
  - `*-แบ่งเขต.pdf` → 5/18 only (constituency vote)
  - `*-บัญชีรายชื่อ.pdf` → 5/18 (บช) only (party-list vote)
  - `*-แบ่งเขต_บัญชีรายชื่อ.pdf` → mixed (both forms)

Filename normalization: `ต.X-...`, `X-001-...`, and slashes/spaces in the original name.


In [7]:
import re

# === Tambol name variants ===
# ECT sometimes mis-spells tambol names in filenames. Map any variant → canonical name from stations.csv.
TAMBOL_ALIASES = {
    # canonical_name: list of known variants seen in filenames
    "ลุ่มลำชี": ["ลุ่มน้ำชี", "ลุ่มลำชี"],
    # add more as you find them...
}

# Build reverse lookup
_VARIANT_TO_CANONICAL = {}
for canonical, variants in TAMBOL_ALIASES.items():
    for v in variants:
        _VARIANT_TO_CANONICAL[v] = canonical


def normalize_tambol_name(raw: str) -> str:
    """Pull a clean tambol name out of various filename patterns."""
    s = raw.strip()
    s = re.sub(r"\.(pdf|PDF)$", "", s)
    s = re.sub(r"^ต\.", "", s)
    s = re.sub(r"^ทต\.", "", s)
    s = re.sub(r"-\d{3}-?", "-", s)
    s = re.sub(r"[-_\s]+(แบ่งเขต|บัญชีรายชื่อ).*$", "", s)
    s = s.replace("_", "").replace(" ", "").strip("- ")
    # Apply alias mapping → canonical name from stations.csv
    return _VARIANT_TO_CANONICAL.get(s, s)


def classify_pdf(path: Path) -> dict:
    """Return what we can infer about this PDF from its filename + location."""
    name = path.name
    base = re.sub(r"\.(pdf|PDF)$", "", name)
    norm = base.replace("_", "/").replace(" ", "")

    # Constituency-level
    if "5/16" in norm or "ส.ส.5_16" in base or "ส.ส. 5_16" in base:
        return {"level": "constituency",
                "form_type": "5_16_party" if "บัญชีรายชื่อ" in base else "5_16",
                "tambol": None}
    if "5/17" in norm or "ส.ส.5_17" in base or "ส.ส. 5_17" in base:
        return {"level": "constituency",
                "form_type": "5_17_party" if "บัญชีรายชื่อ" in base else "5_17",
                "tambol": None}

    # Station-level (5/18) — single-form vs mixed
    has_constit = "แบ่งเขต" in base
    has_party   = "บัญชีรายชื่อ" in base

    if has_constit and has_party:
        form_type = "mixed"
    elif has_constit:
        form_type = "5_18"
    elif has_party:
        form_type = "5_18_party"
    else:
        return {"level": "unknown", "form_type": None, "tambol": None}

    # Special case: เทศบาล file with multiple ทต.
    if base.count("ทต.") >= 2 or (base.startswith("ทต.") and " ทต." in base):
        return {"level": "station", "form_type": form_type,
                "tambol": None, "is_multi_tambol": True}

    return {"level": "station", "form_type": form_type,
            "tambol": normalize_tambol_name(base), "is_multi_tambol": False}


# Sanity check
test_cases = [
    "กะฮาด-001-แบ่งเขต.PDF",
    "ต.กุดน้ำใส-แบ่งเขต_บัญชีรายชื่อ.pdf",
    "ทต.จัตุรัส ทต.บ้านกอก-แบ่งเขต_บัญชีรายชื่อ.pdf",
    "02-แบบ ส.ส.5_16-บัญชีรายชื่อ-นอกเขต.pdf",
    "ต.ลุ่มลำชี แบ่งเขต.PDF",
    "ต.ลุ่มน้ำชี-บัญชีรายชื่อ.pdf",   # ECT typo — should map to ลุ่มลำชี
]
for sample in test_cases:
    result = classify_pdf(Path(sample))
    print(f"  {sample:55s} → {result}")


  กะฮาด-001-แบ่งเขต.PDF                                   → {'level': 'station', 'form_type': '5_18', 'tambol': 'กะฮาด', 'is_multi_tambol': False}
  ต.กุดน้ำใส-แบ่งเขต_บัญชีรายชื่อ.pdf                     → {'level': 'station', 'form_type': 'mixed', 'tambol': 'กุดน้ำใส', 'is_multi_tambol': False}
  ทต.จัตุรัส ทต.บ้านกอก-แบ่งเขต_บัญชีรายชื่อ.pdf          → {'level': 'station', 'form_type': 'mixed', 'tambol': None, 'is_multi_tambol': True}
  02-แบบ ส.ส.5_16-บัญชีรายชื่อ-นอกเขต.pdf                 → {'level': 'constituency', 'form_type': '5_16_party', 'tambol': None}
  ต.ลุ่มลำชี แบ่งเขต.PDF                                  → {'level': 'station', 'form_type': '5_18', 'tambol': 'ลุ่มลำชี', 'is_multi_tambol': False}
  ต.ลุ่มน้ำชี-บัญชีรายชื่อ.pdf                            → {'level': 'station', 'form_type': '5_18_party', 'tambol': 'ลุ่มลำชี', 'is_multi_tambol': False}


## 5 · Coverage analysis (the data-prep insight)

Answer: **What stations do we have data for, vs. what should exist?**

Build a coverage matrix `(tambol × form_type)` showing where data is available, missing, or in mixed-format files. This becomes a deliverable for Component 4 — a transparent account of data limitations.


In [8]:
def build_coverage_report():
    """Match available PDFs against the 341-station expectation."""
    seen = {}
    for p in list(GDRIVE_DIR.rglob("*.pdf")) + list(GDRIVE_DIR.rglob("*.PDF")):
        seen[str(p).lower()] = p
    pdfs = list(seen.values())

    # Classify each available PDF
    available_per_tambol = defaultdict(set)   # tambol -> {form_types}
    multi_tambol_files = []                    # files covering multiple tambols
    constituency_files = {}
    unknown = []

    for pdf in pdfs:
        cls = classify_pdf(pdf)
        if cls["level"] == "constituency":
            constituency_files[cls["form_type"]] = pdf
        elif cls["level"] == "station":
            if cls.get("is_multi_tambol"):
                # Filename can\'t tell us which tambol — nb02 will figure out from page header
                # Pre-emptively mark BOTH forms as available (since file is mixed/may be mixed)
                # for the 2 known municipal tambols (บ้านกอก, หนองบัวโคก in this case)
                multi_tambol_files.append((pdf, cls))
            else:
                tambol = cls["tambol"]
                if cls["form_type"] == "mixed":
                    available_per_tambol[tambol].add("5_18")
                    available_per_tambol[tambol].add("5_18_party")
                else:
                    available_per_tambol[tambol].add(cls["form_type"])
        else:
            unknown.append(pdf)

    # For multi-tambol files: mark both forms as available for KNOWN municipal-area tambols
    # This is heuristic — real assignment happens in nb02 via header parsing
    KNOWN_MULTI_TAMBOLS = ["บ้านกอก", "หนองบัวโคก"]   # ตำบลที่มีสถานีในเขตเทศบาลของอำเภอจัตุรัส
    for pdf, cls in multi_tambol_files:
        for t in KNOWN_MULTI_TAMBOLS:
            if cls["form_type"] == "mixed":
                available_per_tambol[t].add("5_18")
                available_per_tambol[t].add("5_18_party")
            else:
                available_per_tambol[t].add(cls["form_type"])

    # Build per-tambol coverage row
    expected_tambols = stations_df.groupby(["district", "subdistrict"]).size().reset_index(name="n_stations")
    coverage_rows = []
    for _, row in expected_tambols.iterrows():
        tambol = row["subdistrict"]
        forms_present = available_per_tambol.get(tambol, set())
        coverage_rows.append({
            "district":        row["district"],
            "subdistrict":     tambol,
            "n_stations":      row["n_stations"],
            "has_5_18":        "5_18" in forms_present,
            "has_5_18_party":  "5_18_party" in forms_present,
            "completeness":    "complete" if len(forms_present) == 2 else
                            ("partial" if forms_present else "missing"),
        })

    coverage_df = pd.DataFrame(coverage_rows).sort_values(["district", "subdistrict"]).reset_index(drop=True)
    coverage_df.to_csv(EXTERNAL_DIR / "coverage_report.csv", index=False, encoding="utf-8-sig")

    # Summary
    print("=" * 70)
    print(f"COVERAGE SUMMARY for ชัยภูมิ เขต 2")
    print("=" * 70)
    summary = coverage_df.groupby("completeness").agg(
        n_tambols=("subdistrict", "size"),
        n_stations=("n_stations", "sum"),
    )
    print(summary.to_string())
    print()

    total_covered = coverage_df[coverage_df["completeness"] == "complete"]["n_stations"].sum()
    total_partial = coverage_df[coverage_df["completeness"] == "partial"]["n_stations"].sum()
    total_missing = coverage_df[coverage_df["completeness"] == "missing"]["n_stations"].sum()

    print(f"Coverage: {total_covered} / {EXPECTED_STATIONS} stations ({total_covered/EXPECTED_STATIONS*100:.1f}%) full data")
    print(f"          {total_partial} stations have only one of two forms")
    print(f"          {total_missing} stations have no data")
    print()
    print(f"Constituency-level forms found: {sorted(constituency_files.keys())}")
    print(f"Multi-tambol files (เทศบาล): {len(multi_tambol_files)}")
    print(f"Unrecognized files: {len(unknown)}")

    return coverage_df, constituency_files, unknown


coverage_df, constituency_files, unknown = build_coverage_report()
print()
print("Tambol-level detail:")
coverage_df


COVERAGE SUMMARY for ชัยภูมิ เขต 2
              n_tambols  n_stations
completeness                       
complete             20         316
missing               2          25

Coverage: 316 / 341 stations (92.7%) full data
          0 stations have only one of two forms
          25 stations have no data

Constituency-level forms found: ['5_16', '5_16_party', '5_17', '5_17_party']
Multi-tambol files (เทศบาล): 1
Unrecognized files: 0

Tambol-level detail:


,district,subdistrict,n_stations,has_5_18,has_5_18_party,completeness
0,จัตุรัส,กุดน้ำใส,17,True,True,complete
1,จัตุรัส,บ้านกอก,23,True,True,complete
2,จัตุรัส,บ้านขาม,14,True,True,complete
3,จัตุรัส,ละหาน,21,True,True,complete
4,จัตุรัส,ส้มป่อย,15,True,True,complete
5,จัตุรัส,หนองบัวบาน,18,True,True,complete
6,จัตุรัส,หนองบัวโคก,14,True,True,complete
7,จัตุรัส,หนองบัวใหญ่,12,True,True,complete
8,จัตุรัส,หนองโดน,11,True,True,complete
9,ซับใหญ่,ซับใหญ่,14,False,False,missing


## 6 · Build source manifest for nb02

For each station that has data available, build a row pointing to the PDF that contains it. nb02 reads this manifest and OCRs each source file once, dispatching pages to the right station.


In [9]:
def build_source_manifest():
    """
    For each station, find which source PDF contains it.

    Logic:
    - Stations with 'T' in their code are เทศบาล stations.
    - If their อำเภอ has a 'ทต.' file → use the multi-tambol file.
    - Otherwise (no ทต. file in อำเภอ) → T stations are bundled in the regular ต. file.
    """
    seen = {}
    for p in list(GDRIVE_DIR.rglob("*.pdf")) + list(GDRIVE_DIR.rglob("*.PDF")):
        seen[str(p).lower()] = p
    pdfs = list(seen.values())

    by_tambol = defaultdict(dict)              # tambol -> {form_type: (pdf, kind)}
    multi_tambol_pdfs_by_district = defaultdict(dict)  # district -> {form_type: pdf}
    constituency_paths = {}

    for pdf in pdfs:
        cls = classify_pdf(pdf)
        if cls["level"] == "constituency":
            constituency_paths[cls["form_type"]] = pdf
        elif cls["level"] == "station":
            if cls.get("is_multi_tambol"):
                # ทต. file — figure out which district it covers
                # by looking at which folder the file is in
                district = pdf.parent.name.replace("อำเภอ", "")
                if cls["form_type"] == "mixed":
                    multi_tambol_pdfs_by_district[district]["5_18"]       = pdf
                    multi_tambol_pdfs_by_district[district]["5_18_party"] = pdf
                else:
                    multi_tambol_pdfs_by_district[district][cls["form_type"]] = pdf
            else:
                t = cls["tambol"]
                ft = cls["form_type"]
                if ft == "mixed":
                    by_tambol[t]["5_18"]       = (pdf, "mixed")
                    by_tambol[t]["5_18_party"] = (pdf, "mixed")
                else:
                    by_tambol[t][ft] = (pdf, "pure")

    # One row per (station × form_type)
    rows = []
    for _, station in stations_df.iterrows():
        is_municipal = "T" in station["station_code"]   # เทศบาล station
        district = station["district"]
        district_has_multi_file = district in multi_tambol_pdfs_by_district

        for form_type in ("5_18", "5_18_party"):
            # Decision tree:
            # 1. Is this a municipal station AND does the district have a multi-tambol file?
            #    → use the multi-tambol file
            # 2. Otherwise → use the tambol\'s own file (or mark missing)
            if is_municipal and district_has_multi_file and form_type in multi_tambol_pdfs_by_district[district]:
                pdf = multi_tambol_pdfs_by_district[district][form_type]
                rows.append({
                    "station_code":  station["station_code"],
                    "station_no":    station["station_no"],
                    "district":      station["district"],
                    "subdistrict":   station["subdistrict"],
                    "form_type":     form_type,
                    "source_pdf":    str(pdf.relative_to(PROJECT_ROOT)).replace("\\", "/"),
                    "source_kind":   "multi_tambol_mixed",
                })
            else:
                entry = by_tambol.get(station["subdistrict"], {}).get(form_type)
                if entry is None:
                    rows.append({
                        "station_code":  station["station_code"],
                        "station_no":    station["station_no"],
                        "district":      station["district"],
                        "subdistrict":   station["subdistrict"],
                        "form_type":     form_type,
                        "source_pdf":    None,
                        "source_kind":   "missing",
                    })
                else:
                    pdf, kind = entry
                    rows.append({
                        "station_code":  station["station_code"],
                        "station_no":    station["station_no"],
                        "district":      station["district"],
                        "subdistrict":   station["subdistrict"],
                        "form_type":     form_type,
                        "source_pdf":    str(pdf.relative_to(PROJECT_ROOT)).replace("\\", "/"),
                        "source_kind":   kind,
                    })

    # Constituency-level rows
    for form_type in ("5_16", "5_16_party", "5_17", "5_17_party"):
        pdf = constituency_paths.get(form_type)
        rows.append({
            "station_code":  None,
            "station_no":    None,
            "district":      None,
            "subdistrict":   None,
            "form_type":     form_type,
            "source_pdf":    str(pdf.relative_to(PROJECT_ROOT)).replace("\\", "/") if pdf else None,
            "source_kind":   "constituency" if pdf else "missing",
        })

    manifest = pd.DataFrame(rows)
    manifest.to_csv(EXTERNAL_DIR / "source_manifest.csv", index=False, encoding="utf-8-sig")
    return manifest


source_manifest = build_source_manifest()
print(f"Total rows: {len(source_manifest)}")
print()
print("By source kind:")
print(source_manifest["source_kind"].value_counts().to_string())
print()
print("Sample of multi_tambol stations (should be only อ.จัตุรัส, T-suffix):")
multi = source_manifest[source_manifest["source_kind"] == "multi_tambol_mixed"]
print(f"  {len(multi)} rows total")
print(multi[["station_code", "district", "subdistrict", "form_type"]].head(10).to_string(index=False))


Total rows: 686

By source kind:
source_kind
mixed                 406
pure                  204
missing                50
multi_tambol_mixed     22
constituency            4

Sample of multi_tambol stations (should be only อ.จัตุรัส, T-suffix):
  22 rows total
     station_code district subdistrict  form_type
36-02-360601T-001  จัตุรัส     บ้านกอก       5_18
36-02-360601T-001  จัตุรัส     บ้านกอก 5_18_party
36-02-360601T-002  จัตุรัส     บ้านกอก       5_18
36-02-360601T-002  จัตุรัส     บ้านกอก 5_18_party
36-02-360601T-003  จัตุรัส     บ้านกอก       5_18
36-02-360601T-003  จัตุรัส     บ้านกอก 5_18_party
36-02-360601T-004  จัตุรัส     บ้านกอก       5_18
36-02-360601T-004  จัตุรัส     บ้านกอก 5_18_party
36-02-360601T-005  จัตุรัส     บ้านกอก       5_18
36-02-360601T-005  จัตุรัส     บ้านกอก 5_18_party


---

## ✅ When this notebook is done

| File | Rows | Used by |
|---|---|---|
| `data/external/stations.csv` | 341 | nb02, nb04 |
| `data/external/candidates.csv` | 7 | nb02 |
| `data/external/parties.csv` | 57 | nb02 |
| `data/external/reference_constituency.json` | 1 | nb04 (QA) |
| `data/external/coverage_report.csv` | 22 (one per ตำบล) | **nb04 / Component 4 insight** |
| `data/external/source_manifest.csv` | 686 (1 per station × form + 4 const) | nb02 (input) |

## Demo moment for the video 🎯

> "The Drive published by the lecturer covers 19 of 22 ตำบล in our เขต — 316/341 stations.
> Our pipeline transparently identifies the gap and proceeds with what\'s available.
> When we sum our OCR\'d totals, the difference from PBS\'s reported total reflects exactly
> the missing 3 ตำบล — proving both our coverage report and our OCR are correct."
